# Week 2: NumPy & Pandas
### Fall ’26 Data Science Bootcamp - Angelin Ann Jacob
**Brooklyn Bridge Pedestrian Counts**

We will use one dataset throughout this notebook: the [**Brooklyn Bridge Automated Pedestrian Counts Demonstration Project** from NYC OpenData](https://www.nyc.gov/opendata). The dataset contains hourly pedestrian counts together with direction, weather, temperature, precipitation, location, and event information.

The dataset is historical. The current NYC OpenData page is labelled **Historic**; the data was last updated on March 5, 2020.

## How this notebook works

Every topic has two parts, and you will see both labels all the way through:

- **THE STORY**: the question we are asking about the traffic over the bridge, in plain English.


- **THE TOOL**: the NumPy or pandas function we use to answer it.

## Numpy vs Pandas

| **NumPy** | **Pandas** |
|---|---|
| We use NumPy arrays when the data we are working with is primarily numerical and we want fast, compact array operations. | We use pandas when the data is tabular and we need labelled rows, columns, filtering, grouping, and mixed data types. |
| The main object is the `numpy.ndarray`. | The main objects are `Series` and `DataFrame`. |
| Arrays are especially useful for numerical computation and multi-dimensional data. | DataFrames are especially useful for working with real-world tables. |

# Question about the data

**How does pedestrian traffic on the Brooklyn Bridge change across hours, direction, and weather?**

For example:

1. **When** do people cross? (the shape of a typical day)
2. **Which way** do they go more often? (towards Manhattan or towards Brooklyn)
3. **Does weather matter?** (rain versus dry, and each weather category)

Every NumPy and pandas operation below is used to work towards answering one of these.

### Dataset source

NYC OpenData dataset: **Brooklyn Bridge Automated Pedestrian Counts Demonstration Project (Historic)**

Dataset ID: `6fi9-q3ta`

The notebook loads the CSV directly from the NYC OpenData export endpoint.

In [ ]:
import numpy as np
import pandas as pd

DATA_URL = "https://data.cityofnewyork.us/api/v3/views/6fi9-q3ta/export.csv?accessType=DOWNLOAD"
SOURCE_URL = "https://data.cityofnewyork.us/Transportation/Brooklyn-Bridge-Automated-Pedestrian-Counts-Demons/6fi9-q3ta"

### Load the data once

> **THE STORY** - Get one clean, time-ordered table that we reuse for the whole session.
>
> **THE TOOL** - `pd.read_csv`, `pd.to_datetime`, `sort_values`.

In [ ]:
bridge = pd.read_csv(DATA_URL, thousands=",")   # thousands="," keeps values like 1,234 as numbers

# Make the timestamp a real date-time, then put the rows in time order.
bridge["hour_beginning"] = pd.to_datetime(bridge["hour_beginning"], format="mixed")
bridge = bridge.sort_values("hour_beginning").reset_index(drop=True)

bridge.head()

,hour_beginning,location,Pedestrians,Towards Manhattan,Towards Brooklyn,weather_summary,temperature,precipitation,lat,long,events,Location1
0,2017-10-01 00:00:00,Brooklyn Bridge,44,30,14,clear-night,52.0,0.0001,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
1,2017-10-01 01:00:00,Brooklyn Bridge,30,17,13,partly-cloudy-night,53.0,0.0002,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
2,2017-10-01 02:00:00,Brooklyn Bridge,25,13,12,partly-cloudy-night,52.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
3,2017-10-01 03:00:00,Brooklyn Bridge,20,11,9,partly-cloudy-night,51.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
4,2017-10-01 04:00:00,Brooklyn Bridge,18,10,8,partly-cloudy-night,51.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"


If you mix types, NumPy finds a type that can hold everything

For example:

NumPy promotes the integers to floats: float64

So:
1      → 1.0
2      → 2.0
3.5    → 3.5

because an integer array couldn't represent 3.5.

In [ ]:
np.array([1, 2, 3.5])
type(np.array([1, 2, 3.5])[0])

numpy.float64

Another example:

NumPy may choose a string dtype because it needs one dtype capable of representing all the elements.

In [ ]:
np.array([1, 2, "hello"])
type(np.array([1, 2, "hello"])[0])

numpy.str_

In [ ]:
l = [1, "Brooklyn Bridge", 1.23]
l

[1, 'Brooklyn Bridge', 1.23]

NumPy determines the data type (dtype) of an array, it usually looks at the values you give it and chooses a compatible NumPy type.


eg: NumPy sees that all the values are Python integers → chooses an integer dtype.

This next cell makes sure the count columns are stored as numbers rather than text, so every later reference uses the correct datatype. You should see numeric types (`int64` or `float64`) and few or no missing values.

In [ ]:
count_cols = ["Pedestrians", "Towards Manhattan", "Towards Brooklyn"]
for col in count_cols:
    bridge[col] = pd.to_numeric(bridge[col], errors="coerce")
    # errors="coerce":  If datatype cannot convert to a number, replace it with NaN instead of throwing an error.
    # [1, NaN, 1.23]

print(bridge[count_cols + ["temperature"]].dtypes)

Pedestrians            int64
Towards Manhattan      int64
Towards Brooklyn       int64
temperature          float64
dtype: object


## 1.1 Why arrays, not lists?

> **THE STORY** - We have thousands of hourly counts and temperatures. We want to do arithmetic on all of them at once, not one at a time.
>
> **THE TOOL** - `.to_numpy()`, which turns a pandas column into a NumPy array (whole-array arithmetic follows in 1.2).

Lists can contain mixed Python objects.

vs


NumPy arrays are designed for numerical work and normally store values of one data type. This gives NumPy compact storage and fast vectorised operations implemented in optimised code.

For this notebook, think of the difference this way: pandas helps us understand the **table**, while NumPy helps us manipulate the **numbers inside the table**.

### Pull a few columns out of the table as NumPy arrays

In [ ]:
pedestrians = bridge["Pedestrians"].to_numpy(copy=True)
towards_manhattan = bridge["Towards Manhattan"].to_numpy(copy=True)
towards_brooklyn = bridge["Towards Brooklyn"].to_numpy(copy=True)
temperature = bridge["temperature"].to_numpy(copy=True)

print(type(pedestrians))
print(pedestrians[:10])

<class 'numpy.ndarray'>
[ 44  30  25  20  18  16 113 199 261 845]


`copy=True` gives us independent arrays, so we can experiment freely without touching the table.

## 1.2 Arrays and array construction

> **THE STORY** - Convert temperatures from °F to °C, check that the two directions add up to the total, and prepare empty 24-hour profiles to fill later.
>
> **THE TOOL** - `np.array`, `np.zeros`, `np.ones`, `np.random.random`, `np.arange`, `np.linspace`, `np.vstack`, `np.hstack`.

A NumPy array is a grid of values of a common type. We can create one directly from a Python list with `np.array()`.

In [ ]:
a = np.array(pedestrians[:5])
print(a)
print(a.shape)

[44 30 25 20 18]
(5,)


The useful part is that arithmetic works on the whole array at once. For example, Fahrenheit temperatures can be converted to Celsius without writing a loop.

In [ ]:
sample_f = np.array(temperature[:5])
sample_c = (sample_f - 32) * 5 / 9
print(sample_f)
print(sample_c)

[52. 53. 52. 51. 51.]
[11.11111111 11.66666667 11.11111111 10.55555556 10.55555556]


A useful sanity check, also without a loop: the two directions should add up to the total.

In [ ]:
direction_total = towards_manhattan[:5] + towards_brooklyn[:5]
print("direction total:", direction_total)
print("recorded total: ", pedestrians[:5])

direction total: [44 30 25 20 18]
recorded total:  [44 30 25 20 18]


`np.zeros`, `np.ones`, and `np.random.random` are useful when we want NumPy to initialise an array for us.

In [ ]:
empty_hour_profile = np.zeros(24)
weights = np.ones(24)
random_day = np.random.random(24)

print(empty_hour_profile)
print(weights)
print(random_day)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[0.4846724  0.70132191 0.82780721 0.92528822 0.65774979 0.78542569
 0.63089071 0.61318021 0.43616872 0.63574192 0.53929686 0.93308614
 0.21406456 0.9061654  0.2613368  0.31165212 0.62631212 0.65049661
 0.37654499 0.23724951 0.90026739 0.59490534 0.6244937  0.84717566]


Here `zeros` gives us 24 placeholders: one for each hour of a day. Later, we can fill those positions with observed hourly averages.

### Creating values with `arange` and `linspace`

In [ ]:
hours = np.arange(0, 24)
print(hours)

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]


In [ ]:
bins_by_step = np.arange(0, pedestrians.max() + 500, 500)
bins_by_count = np.linspace(0, pedestrians.max(), 10)

print(bins_by_step)
print(bins_by_count)

[   0  500 1000 1500 2000 2500 3000 3500 4000 4500]
[   0.          481.11111111  962.22222222 1443.33333333 1924.44444444
 2405.55555556 2886.66666667 3367.77777778 3848.88888889 4330.        ]


`arange` is controlled by a **step size**; `linspace` is controlled by the **number of values**. Here they give two ways to construct possible bins for thinking about pedestrian volume.

### Stacking directional counts

> Matrix like operations are made simpler in numpy!


- `direction_rows`: row 0 = Manhattan, row 1 = Brooklyn
- `direction_columns`: col 0 = Manhattan, col 1 = Brooklyn, so each row is one `(manhattan, brooklyn)` pair



In [ ]:
direction_rows = np.vstack((towards_manhattan[:5], towards_brooklyn[:5]))
direction_columns = np.hstack((
    towards_manhattan[:5, np.newaxis],
    towards_brooklyn[:5, np.newaxis]
))

print("vstack shape:", direction_rows.shape)
print(direction_rows)
print("hstack shape:", direction_columns.shape)
print(direction_columns)

vstack shape: (2, 5)
[[30 17 13 11 10]
 [14 13 12  9  8]]
hstack shape: (5, 2)
[[30 14]
 [17 13]
 [13 12]
 [11  9]
 [10  8]]


For a two-dimensional array we provide one index per dimension.

In [ ]:
print(direction_columns)
print(direction_columns[0, 0])
print(direction_columns[0][0])
print(direction_columns[-1, -1])

[[30 14]
 [17 13]
 [13 12]
 [11  9]
 [10  8]]
30
30
8


The two forms `a[i, j]` and `a[i][j]` reach the same element here, although comma-separated indexing is the usual NumPy style.

The direction data naturally gives us two related arrays: pedestrians walking **towards Manhattan** and **towards Brooklyn**. Stacking lets us treat those two series as a small matrix.

## Exercise 1 - Temperatures


Using the temperature readings, create a NumPy array and convert the temperatures from Fahrenheit to Celsius, and append this to the `direction_columns`. Do not use a Python loop. Print first 5 elements of the `direction_columns`.

In [ ]:
# Your code here

## 1.3 What kind of data are we holding?

> **THE STORY** - Before trusting any column of numbers, ask what it is: how many dimensions, how many values, what type?
>
> **THE TOOL** - `ndim`, `shape`, `size`, `dtype`, and `np.random.randint` for spot checks.

Every NumPy array has attributes that describe its structure: `ndim`, `shape`, `size`, and `dtype`.

In [ ]:
print("ndim:", pedestrians.ndim)
print("shape:", pedestrians.shape)
print("size:", pedestrians.size)
print("dtype:", pedestrians.dtype)

ndim: 1
shape: (16057,)
size: 16057
dtype: int64


We can ask the same questions of the direction matrix.

In [ ]:
print("direction matrix ndim:", direction_columns.ndim)
print("direction matrix shape:", direction_columns.shape)
print("direction matrix size:", direction_columns.size)

direction matrix ndim: 2
direction matrix shape: (5, 2)
direction matrix size: 10


For a real analysis, we would not silently overwrite the original dataset. That is why we pulled the arrays out with `copy=True`: changing them never changes the `bridge` table.

### Random spot checks with `np.random.randint`

In [ ]:
rng_indices = np.random.randint(0, len(pedestrians), size=5)
print("random indices:", rng_indices)
print("random counts:", pedestrians[rng_indices])

random indices: [  637  1566 10643 11447 11664]
random counts: [2753  101 1465  109    3]


This is a useful habit when inspecting a large table: choose a few random records rather than looking only at the first rows.

## Exercise 2 - Indexing

Use random indexing ONLY to report any 10 pedestrian counts in the NumPy array and its respective weather in that hour. Then create a two-column array containing the first five values for each direction and retrieve the Manhattan-bound count from row 0.

In [ ]:
# Your code here

## 1.4 Array slicing: accessing subarrays

> **THE STORY** - Pull out one full day and sample the same hour on every day.
>
> **THE TOOL** - Slices `a[start:stop:step]` and 2-D slices `a[:, 0]`.

Slicing follows the pattern `start:stop:step`. The stop position is excluded.

In [ ]:
first_day = pedestrians[:24]
first_day

array([  44,   30,   25,   20,   18,   16,  113,  199,  261,  845, 2478,
       3006, 2263, 2616, 2463, 2245, 2764, 2247, 2142,  899,  481,  275,
        133,  104])

A step lets us skip values. Because the data are in chronological order, every 24th value gives one reading at the same hour of the day (as long as no hours are missing).

In [ ]:
midnight_samples = pedestrians[::24]
midnight_samples[:10]

array([ 44,  60,  45,  28,  41,  50,  88, 145, 112,  29])

For two-dimensional arrays, each dimension gets its own slice. Rows and columns can be selected separately.

In [ ]:
print(direction_columns)
print(direction_columns[:3, :2])
print("Manhattan column:", direction_columns[:, 0])
print("Brooklyn column: ", direction_columns[:, 1])

[[30 14]
 [17 13]
 [13 12]
 [11  9]
 [10  8]]
[[30 14]
 [17 13]
 [13 12]]
Manhattan column: [30 17 13 11 10]
Brooklyn column:  [14 13 12  9  8]


## 1.5 Reshaping and axes

> **THE STORY** - Two months of hourly counts are one long list.
>
> To compare days we need a grid: one row per day, one column per hour. Then we collapse that grid into a typical day.
>
> **THE TOOL** - `.reshape`, `.sum(axis=...)`, `.mean(axis=...)`, `.copy()`.

Reshaping changes the arrangement of values without changing the number of values. The source and target arrays must contain the same number of elements.

In [ ]:
grid = np.arange(1, 13).reshape((3, 4))
print(grid)
print(grid.shape)

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
(3, 4)


The Brooklyn Bridge data can be arranged into a day-by-hour matrix **only after we have checked that the selected period contains a complete sequence of hourly observations**.

In [ ]:
study_start = pd.Timestamp("2017-10-07")
study_end = pd.Timestamp("2017-12-04")
# 7 Oct 2017 → 4 Dec 2017
study = bridge[(bridge["hour_beginning"] >= study_start) & (bridge["hour_beginning"] < study_end)].copy()

ped_study = study["Pedestrians"].to_numpy(copy=True)
print("study rows:", len(ped_study))
print("first timestamp:", study["hour_beginning"].min())
print("last timestamp:", study["hour_beginning"].max())

study rows: 1392
first timestamp: 2017-10-07 00:00:00
last timestamp: 2017-12-03 23:00:00


If the study window has exactly 1,392 hourly observations, that is 58 days × 24 hours.

In [ ]:
expected_rows = (study_end - study_start).days * 24
print("rows expected:", expected_rows)
print("rows found:   ", len(ped_study))
print("complete window:", len(ped_study) == expected_rows)

rows expected: 1392
rows found:    1392
complete window: True


In [ ]:
day_by_hour = ped_study.reshape((study_end - study_start).days, 24)
print(day_by_hour.shape)
day_by_hour[:2]

(58, 24)


array([[  88,   64,   29,   10,    9,   21,   84,  244,  378,  989, 1737,
        2602, 2983, 3191, 3251, 3633, 3388, 3373, 2742, 1132,  665,  373,
         356,  268],
       [ 145,   57,   46,   11,   26,   24,   41,  160,  216,  164,  479,
         882, 1504, 1923, 2599, 2606, 2531, 2485, 1745,  783,  589,  453,
         311,  213]])

If the check above had printed `False`, this reshape would fail with a `ValueError`: the number of values must match.

#### Axis 0 vs axis 1

#### Axes are the questions you ask

The same array can answer different questions. The axis you collapse decides which one:

```python
DAY, HOUR = 0, 1   # day_by_hour has shape (days, hours)

typical_day = day_by_hour.mean(axis=DAY)    # "What does an average day look like?"
daily_total = day_by_hour.sum(axis=HOUR)    # "How busy was each day?"

rush_hour   = typical_day.argmax()          # busiest hour of the day, on average
busiest_day = daily_total.argmax()          # position of the busiest day
```

**Why this matters in data analysis**
- `axis=0` means nothing to a reader, but we have now encoded `axis=DAY` which allows easier understandings of computations across a direction/axis!
- **It scales.** With more dimensions (day × hour × direction), encoded axes keep the code readable.

In [ ]:
DAY, HOUR = 0, 1

print("mean across days (axis=0):", day_by_hour.mean(axis=DAY)[:5])
print("sum across hours (axis=1):", day_by_hour.sum(axis=HOUR)[:5])

mean across days (axis=0): [54.37931034 21.36206897 10.96551724  6.53448276  4.94827586]
sum across hours (axis=1): [31610 19993  8985 20821 15755]


The axis tells NumPy **which direction to collapse**. For our `(58, 24)` array, `axis=0` collapses the 58 days and leaves 24 hourly positions (a **typical day**); `axis=1` collapses the 24 hours and leaves 58 daily totals.

## 1.6 Combining arrays and making decisions

> **THE STORY** - Join the two directions into one structure, then label each hour with a decision: which direction dominates, and is it busy or quiet?
>
> **THE TOOL** - `np.concatenate`, `np.vstack`, `np.hstack`, `np.where`.

Concatenation combines existing arrays. In this notebook, the two directions give us a natural pair of arrays to combine.

In [ ]:
manhattan_small = towards_manhattan[:5]
brooklyn_small = towards_brooklyn[:5]

combined_1d = np.concatenate([manhattan_small, brooklyn_small])
combined_rows = np.vstack([manhattan_small, brooklyn_small])
combined_columns = np.hstack([
    manhattan_small[:, np.newaxis],
    brooklyn_small[:, np.newaxis]
])

print("concatenate:", combined_1d)
print(combined_rows)
print("vstack shape:", combined_rows.shape)
print(combined_columns)
print("hstack shape:", combined_columns.shape)

print(combined_rows == combined_columns.T)
print((combined_rows == combined_columns.T).shape)

concatenate: [30 17 13 11 10 14 13 12  9  8]
[[30 17 13 11 10]
 [14 13 12  9  8]]
vstack shape: (2, 5)
[[30 14]
 [17 13]
 [13 12]
 [11  9]
 [10  8]]
hstack shape: (5, 2)
[[ True  True  True  True  True]
 [ True  True  True  True  True]]
(2, 5)


## Bonus 1 - Why are the dimensions (2,5)

In [ ]:
# Your code & explaination here

`np.where` lets us make an element-by-element decision. First, which direction has more pedestrians in each hour?

In [ ]:
dominant_direction = np.where(towards_manhattan >= towards_brooklyn, "Manhattan", "Brooklyn")
dominant_direction[:10]

array(['Manhattan', 'Manhattan', 'Manhattan', 'Manhattan', 'Manhattan',
       'Manhattan', 'Manhattan', 'Brooklyn', 'Manhattan', 'Manhattan'],
      dtype='<U9')

We can also label an hour as `busy` when its count is at or above the overall median (`np.median` gives the middle value) and `quieter` otherwise.

In [ ]:
threshold = np.median(pedestrians)
traffic_label = np.where(pedestrians >= threshold, "busy", "quieter")
traffic_label[:20]

array(['quieter', 'quieter', 'quieter', 'quieter', 'quieter', 'quieter',
       'quieter', 'quieter', 'busy', 'busy', 'busy', 'busy', 'busy',
       'busy', 'busy', 'busy', 'busy', 'busy', 'busy', 'busy'],
      dtype='<U7')

## 1.7 Could NumPy read the CSV by itself?

> **THE STORY** - Could we have skipped pandas and loaded the file with NumPy alone?
>
> **THE TOOL** - `np.genfromtxt`. It works well for clean numeric columns.

Yes, for a simple numeric column NumPy does the job, and the result is identical to the pandas route.

The difference is the **effort and the risk**:

| | NumPy `genfromtxt` | pandas `read_csv` |
|---|---|---|
| Header row | skip it yourself | detected automatically |
| Choosing a column | by position (`usecols=[2]`) | by name (`"Pedestrians"`) |
| Timestamps and text | no built-in support | dates and text handled |
| Mixed columns | one data type per array | each column keeps its own type |

**Takeaway:** NumPy is built for fast maths on numeric arrays, and pandas is built for loading and labelling messy tables. They work together: load and clean with pandas, then pass NumPy the clean array with `.to_numpy()`.

In [ ]:
bridge.to_csv("brooklyn_bridge_clean.csv", index=False)

raw_pedestrians = np.genfromtxt(
    "brooklyn_bridge_clean.csv",
    delimiter=",",
    skip_header=1,
    usecols=[2],
    dtype=str
)

raw_pedestrians = raw_pedestrians.astype("float")
print(raw_pedestrians[:10])
print(raw_pedestrians.dtype)

[ 44.  30.  25.  20.  18.  16. 113. 199. 261. 845.]
float64


Notice what we had to do: skip the header row ourselves, pick the column by **position** instead of by name, and convert text to numbers by hand. Here we deliberately load only one numeric column; handling all of the mixed columns this way would be much less convenient than using a DataFrame.

# Part 2 - pandas: the table itself

Pandas is designed for structured tables. Our Brooklyn Bridge dataset is a good example because one row represents one hourly observation while the columns mix timestamps, counts, weather categories, and location information.

The **DataFrame** is a two-dimensional labelled data structure. Each column can have its own datatype, while the row labels form an index. A single column of a DataFrame is a **Series**.

## 2.1 Getting to know your data

> **THE STORY** - What did we just load? How big is it, what are the columns called, what type is each, and what are typical values?
>
> **THE TOOL** - `.head()`, `.tail()`, `.shape`, `.info()`, `.columns`, `.describe()`, `.dtypes`.

### See the first 10 entries

In [ ]:
bridge.head(10)

,hour_beginning,location,Pedestrians,Towards Manhattan,Towards Brooklyn,weather_summary,temperature,precipitation,lat,long,events,Location1
0,2017-10-01 00:00:00,Brooklyn Bridge,44,30,14,clear-night,52.0,0.0001,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
1,2017-10-01 01:00:00,Brooklyn Bridge,30,17,13,partly-cloudy-night,53.0,0.0002,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
2,2017-10-01 02:00:00,Brooklyn Bridge,25,13,12,partly-cloudy-night,52.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
3,2017-10-01 03:00:00,Brooklyn Bridge,20,11,9,partly-cloudy-night,51.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
4,2017-10-01 04:00:00,Brooklyn Bridge,18,10,8,partly-cloudy-night,51.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
5,2017-10-01 05:00:00,Brooklyn Bridge,16,12,4,partly-cloudy-night,50.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
6,2017-10-01 06:00:00,Brooklyn Bridge,113,88,25,partly-cloudy-night,49.0,0.0001,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
7,2017-10-01 07:00:00,Brooklyn Bridge,199,72,127,clear-day,49.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
8,2017-10-01 08:00:00,Brooklyn Bridge,261,133,128,clear-day,50.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
9,2017-10-01 09:00:00,Brooklyn Bridge,845,504,341,clear-day,54.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"


### Print the last elements of the dataset

In [ ]:
bridge.tail()

,hour_beginning,location,Pedestrians,Towards Manhattan,Towards Brooklyn,weather_summary,temperature,precipitation,lat,long,events,Location1
16052,2019-12-31 19:00:00,Brooklyn Bridge,11,9,2,cloudy,43.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
16053,2019-12-31 20:00:00,Brooklyn Bridge,15,14,1,cloudy,43.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
16054,2019-12-31 21:00:00,Brooklyn Bridge,12,0,12,cloudy,42.0,0.0036,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
16055,2019-12-31 22:00:00,Brooklyn Bridge,10,9,1,cloudy,42.0,0.0005,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
16056,2019-12-31 23:00:00,Brooklyn Bridge,2,0,2,cloudy,42.0,0.0004,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"


### How many observations and columns?

Use `.shape` first. Pay attention to which number represents rows and which represents columns.

In [ ]:
bridge.shape

(16057, 12)

### Another way

In [ ]:
bridge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16057 entries, 0 to 16056
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   hour_beginning     16057 non-null  datetime64[ns]
 1   location           16057 non-null  object        
 2   Pedestrians        16057 non-null  int64         
 3   Towards Manhattan  16057 non-null  int64         
 4   Towards Brooklyn   16057 non-null  int64         
 5   weather_summary    16041 non-null  object        
 6   temperature        16041 non-null  float64       
 7   precipitation      16041 non-null  float64       
 8   lat                16057 non-null  float64       
 9   long               16057 non-null  float64       
 10  events             1124 non-null   object        
 11  Location1          16057 non-null  object        
dtypes: datetime64[ns](1), float64(4), int64(3), object(4)
memory usage: 1.5+ MB


### What are the columns called?

In [ ]:
bridge.columns

Index(['hour_beginning', 'location', 'Pedestrians', 'Towards Manhattan',
       'Towards Brooklyn', 'weather_summary', 'temperature', 'precipitation',
       'lat', 'long', 'events', 'Location1'],
      dtype='object')

### What are the headline statistics for the numerical columns?

Use `.describe()` and compare the **mean** and **median** (50%) of `Pedestrians`. What does the difference suggest about the distribution of hourly traffic?

In [ ]:
bridge.describe()

,hour_beginning,Pedestrians,Towards Manhattan,Towards Brooklyn,temperature,precipitation,lat,long
count,16057,16057.000000,16057.000000,16057.000000,16041.000000,16041.000000,1.605700e+04,1.605700e+04
mean,2018-11-22 23:25:25.913931776,687.106309,334.772436,352.286853,53.213952,0.004617,4.070816e+01,-7.399951e+01
min,2017-10-01 00:00:00,0.000000,0.000000,0.000000,2.000000,0.000000,4.070816e+01,-7.399951e+01
25%,2018-03-17 06:00:00,16.000000,9.000000,5.000000,39.000000,0.000000,4.070816e+01,-7.399951e+01
50%,2019-01-31 12:00:00,227.000000,112.000000,111.000000,53.000000,0.000000,4.070816e+01,-7.399951e+01
75%,2019-07-17 18:00:00,1254.000000,611.000000,632.000000,69.000000,0.000000,4.070816e+01,-7.399951e+01
max,2019-12-31 23:00:00,4330.000000,3657.000000,2872.000000,97.000000,0.680400,4.070816e+01,-7.399951e+01
std,NaN,862.244605,417.807545,456.624509,18.036800,0.023400,5.485561e-12,1.277596e-11


### What are the column datatypes?

In [ ]:
bridge.dtypes

,0
hour_beginning,datetime64[ns]
location,object
Pedestrians,int64
Towards Manhattan,int64
Towards Brooklyn,int64
weather_summary,object
temperature,float64
precipitation,float64
lat,float64
long,float64


In the setup we converted the timestamp to a pandas date-time so that we can work with time safely. Here is the proof:

In [ ]:
bridge["hour_beginning"].dtype

dtype('<M8[ns]')

In [ ]:
# Your code here

## 2.2 Counting things

> **THE STORY** - How many people were recorded in total, and how common is each kind of weather?
>
> **THE TOOL** - `.sum()` and `.value_counts()` on a column.

In [ ]:
total_pedestrians = bridge["Pedestrians"].sum()
total_pedestrians

np.int64(11032866)

In [ ]:
bridge["weather_summary"].value_counts()

,count
weather_summary,
clear-night,3750
partly-cloudy-day,3169
clear-day,3127
partly-cloudy-night,2503
cloudy,2377
rain,920
snow,93
fog,76
sleet,14


## 2.3 Sorting, filtering, and finding one row

> **THE STORY** - Which hours were busiest? Do rainy hours look different from dry ones? What exactly happened in the single busiest hour?
>
> **THE TOOL** - `.sort_values()`, selecting columns with `[[...]]`, a True/False filter such as `df[df["col"] > 0]`, `.idxmax()`, and `.loc`.

### Which hourly observations were the busiest?

In [ ]:
bridge.sort_values(by="Pedestrians", ascending=False).head(10)

,hour_beginning,location,Pedestrians,Towards Manhattan,Towards Brooklyn,weather_summary,temperature,precipitation,lat,long,events,Location1
659,2017-10-28 11:00:00,Brooklyn Bridge,4330,2150,2180,clear-day,64.0,0.0001,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
10767,2019-05-25 15:00:00,Brooklyn Bridge,4286,2121,2165,partly-cloudy-day,68.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
4357,2018-03-31 13:00:00,Brooklyn Bridge,4210,2062,2148,clear-day,51.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
7311,2019-01-01 15:00:00,Brooklyn Bridge,4141,2575,1566,partly-cloudy-day,49.0,0.0000,40.708164,-73.999509,New Year's Day,"(40.7081639691088, -73.9995087014816)"
15976,2019-12-28 15:00:00,Brooklyn Bridge,4108,1236,2872,clear-day,50.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
4356,2018-03-31 12:00:00,Brooklyn Bridge,4058,1980,2078,clear-day,50.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
661,2017-10-28 13:00:00,Brooklyn Bridge,4055,1560,2495,clear-day,67.0,0.0028,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"
7312,2019-01-01 16:00:00,Brooklyn Bridge,4012,2012,2000,partly-cloudy-day,47.0,0.0000,40.708164,-73.999509,New Year's Day,"(40.7081639691088, -73.9995087014816)"
7310,2019-01-01 14:00:00,Brooklyn Bridge,4008,1994,2014,partly-cloudy-day,52.0,0.0000,40.708164,-73.999509,New Year's Day,"(40.7081639691088, -73.9995087014816)"
4359,2018-03-31 15:00:00,Brooklyn Bridge,4001,1975,2026,clear-day,54.0,0.0000,40.708164,-73.999509,NaN,"(40.7081639691088, -73.9995087014816)"


### Show only the pedestrian counts and timestamps for the five busiest observations

In [ ]:
bridge.sort_values(by="Pedestrians", ascending=False)[["hour_beginning", "Pedestrians", "weather_summary"]].head(5)

,hour_beginning,Pedestrians,weather_summary
659,2017-10-28 11:00:00,4330,clear-day
10767,2019-05-25 15:00:00,4286,partly-cloudy-day
4357,2018-03-31 13:00:00,4210,clear-day
7311,2019-01-01 15:00:00,4141,partly-cloudy-day
15976,2019-12-28 15:00:00,4108,clear-day


### Rainy hours versus dry hours

A filter keeps only the rows where a condition is `True`. Here "wet" means any recorded precipitation, even tiny amounts.

In [ ]:
wet_hours = bridge[bridge["precipitation"] > 0]
dry_hours = bridge[bridge["precipitation"] == 0]

print("wet observations:", len(wet_hours))
print("dry observations:", len(dry_hours))
print("mean during wet hours:", wet_hours["Pedestrians"].mean())
print("mean during dry hours:", dry_hours["Pedestrians"].mean())

wet observations: 3717
dry observations: 12324
mean during wet hours: 541.1676082862524
mean during dry hours: 732.0042194092827


### Find a single timestamp with `.loc`

In [ ]:
busiest_index = bridge["Pedestrians"].idxmax()
bridge.loc[busiest_index]

,659
hour_beginning,2017-10-28 11:00:00
location,Brooklyn Bridge
Pedestrians,4330
Towards Manhattan,2150
Towards Brooklyn,2180
weather_summary,clear-day
temperature,64.0
precipitation,0.0001
lat,40.708164
long,-73.999509


## 2.4 Renaming columns

Column names containing spaces are awkward for attribute-style access. Once a column has a simple Python identifier, dot access `df.column` can be convenient. `rename` returns a **new** table and leaves the original untouched.

In [ ]:
analysis = bridge.rename(columns={
    "Towards Manhattan": "towards_manhattan",
    "Towards Brooklyn": "towards_brooklyn",
    "Pedestrians": "pedestrians"
})

print(analysis.pedestrians.head())
print("original columns unchanged:", list(bridge.columns[2:5]))

0    44
1    30
2    25
3    20
4    18
Name: pedestrians, dtype: int64
original columns unchanged: ['Pedestrians', 'Towards Manhattan', 'Towards Brooklyn']


## 2.5 Grouping: which hour of the day is busiest?

> **THE STORY** - When do people cross? Compare all the 3 PM hours with all the other 3 PM hours, and so on for every hour of the day. And does weather change the crowd?
>
> **THE TOOL** - `.dt.hour`, `.groupby(...)`, `.mean()`, `.agg([...])`, `.describe()`, `.sort_values()`.

In [ ]:
analysis["hour"] = analysis["hour_beginning"].dt.hour
hourly_mean = analysis.groupby("hour").pedestrians.mean()
hourly_mean

,pedestrians
hour,
0,34.038864
1,11.637313
2,5.578475
3,3.330344
4,2.792227
5,20.636771
6,87.328849
7,184.588939
8,309.765321


A group-by answers the question by comparing all observations that share the same hour of day.

### Sort the hourly profile

In [ ]:
hourly_mean.sort_values(ascending=False)

,pedestrians
hour,
15,1838.499253
14,1783.831091
16,1753.310912
13,1723.724963
12,1648.055306
11,1493.902840
17,1421.626308
18,1066.028401
10,1057.479821


### Grouping with several statistics

The same idea, grouped by weather instead of by hour.

In [ ]:
weather_stats = analysis.groupby("weather_summary").pedestrians.agg(["mean", "min", "max"])
weather_stats

,mean,min,max
weather_summary,,,
clear-day,1386.569875,0,4330
clear-night,102.823467,0,1779
cloudy,541.755574,0,3894
fog,234.473684,0,1321
partly-cloudy-day,1422.154307,0,4286
partly-cloudy-night,94.065521,0,1522
rain,256.165217,0,2727
sleet,117.928571,0,404
snow,195.473118,0,1561


### Full spread with `describe()`

In [ ]:
analysis.groupby("weather_summary").pedestrians.describe()

,count,mean,std,min,25%,50%,75%,max
weather_summary,,,,,,,,
clear-day,3127.0,1386.569875,861.890079,0.0,611.50,1401.0,1982.5,4330.0
clear-night,3750.0,102.823467,206.543811,0.0,2.00,19.0,94.0,1779.0
cloudy,2377.0,541.755574,728.429395,0.0,9.00,143.0,948.0,3894.0
fog,76.0,234.473684,307.735795,0.0,15.00,110.0,276.5,1321.0
partly-cloudy-day,3169.0,1422.154307,844.930127,0.0,699.00,1433.0,2008.0,4286.0
partly-cloudy-night,2503.0,94.065521,173.389095,0.0,3.00,23.0,97.5,1522.0
rain,920.0,256.165217,421.571020,0.0,10.00,67.5,311.0,2727.0
sleet,14.0,117.928571,157.850204,0.0,7.25,28.0,254.5,404.0
snow,93.0,195.473118,292.630818,0.0,16.00,77.0,258.0,1561.0


## Exercise 3 - Busiest hour

Create a Series containing the **mean pedestrian count for each hour of the day**. Sort it from busiest hour to quietest hour and report the hour at the top.

In [ ]:
# Your code here

## 2.6 `apply()` with a `lambda`

> **THE STORY** - Weather labels are not very reader-friendly. Make a display version without changing the raw label.
>
> **THE TOOL** - `.apply(...)` with a small `lambda` function, and `.fillna(...)`.

Weather labels are stored as text. We can use a small lambda function with `apply()` to make the first letter uppercase.

In [ ]:
capitalizer = lambda q: " ".join(q.split("-")).capitalize()
capitalizer = lambda q: q.replace("-", " ")
analysis["weather_readable"] = analysis["weather_summary"].fillna("unknown").apply(capitalizer)
analysis[["weather_summary", "weather_readable"]].head(10)

,weather_summary,weather_readable
0,clear-night,clear night
1,partly-cloudy-night,partly cloudy night
2,partly-cloudy-night,partly cloudy night
3,partly-cloudy-night,partly cloudy night
4,partly-cloudy-night,partly cloudy night
5,partly-cloudy-night,partly cloudy night
6,partly-cloudy-night,partly cloudy night
7,clear-day,clear day
8,clear-day,clear day
9,clear-day,clear day


## 2.7 Missing values

> **THE STORY** - Can we trust every row? What links the missing values? Hint: check which columns have missing values and whether they go missing together.
>
> **THE TOOL** - `.isna()`, `.notna()`, `.dropna()`, `.fillna()`.

`isna()` marks missing values as `True`, and `.sum()` counts them. `dropna()` with no arguments removes a row if **any** column is missing.

In [ ]:
print(analysis.isna().sum())          # missing values per column

hour_beginning           0
location                 0
pedestrians              0
towards_manhattan        0
towards_brooklyn         0
weather_summary         16
temperature             16
precipitation           16
lat                      0
long                     0
events               14933
Location1                0
hour                     0
weather_readable         0
dtype: int64


In [ ]:
print("rows with an event:", analysis["events"].notna().sum(), "of", len(analysis))

rows with an event: 1124 of 16057


In [ ]:
weather_cols = ["weather_summary", "temperature", "precipitation"]

event_but_no_weather = analysis[analysis["events"].notna() & analysis[weather_cols].isna().any(axis=1)]
print(len(event_but_no_weather))                      # 4
event_but_no_weather[["hour_beginning", "events"] + weather_cols]

4


,hour_beginning,events,weather_summary,temperature,precipitation
3866,2018-03-11 02:00:00,Daylight Saving Time starts,NaN,NaN,NaN
5376,2018-05-13 00:00:00,Mother's Day,NaN,NaN,NaN
8930,2019-03-10 02:00:00,Daylight Saving Time starts,NaN,NaN,NaN
14642,2019-11-03 01:00:00,Daylight Saving Time ends,NaN,NaN,NaN


In [ ]:
print("before dropna():        ", analysis["events"].notna().sum())
print("after dropna():        ", len(analysis.dropna()))
print("after dropna(subset=): ", len(analysis.dropna(subset=["weather_summary", "temperature"])))



# fillna on a copy of the column, so the table itself is unchanged
analysis["events"].fillna("no event").value_counts().head()

before dropna():         1124
after dropna():         1120
after dropna(subset=):  16041


,count
events,
no event,14933
Halloween,48
Christmas Eve,48
Daylight Saving Time ends,48
Veterans Day,48


## 2.8 New columns: `pd.cut`, `assign`

> **THE STORY** - Temperature is a number, but people think in "freezing" and "warm". Turn the number into a label, then compare traffic across labels.
>
> **THE TOOL** - `pd.cut`, `.assign()`.

`pd.cut` sorts numbers into labelled bins. The right edge of each bin is included, so 32°F falls in "freezing". Missing temperatures stay `NaN`. The thresholds assume °F.


In [ ]:
analysis["temp_band"] = pd.cut(
    analysis["temperature"],
    bins=[-np.inf, 32, 50, 70, np.inf],
    labels=["freezing", "cold", "mild", "warm"],
)

In [ ]:
analysis.groupby("temp_band", observed=True).agg(
    hours=("pedestrians", "count"),
    mean_pedestrians=("pedestrians", "mean"),
    mean_temp_f=("temperature", "mean"),
).round(1)

,hours,mean_pedestrians,mean_temp_f
temp_band,,,
freezing,1988,266.0,25.0
cold,5407,496.6,40.9
mild,5182,700.5,60.7
warm,3464,1209.2,77.4


In [ ]:
# assign: add a column without a separate assignment line
analysis = analysis.assign(temp_c=(analysis["temperature"] - 32) * 5 / 9)

In [ ]:
analysis["temp_c_band"] = pd.cut(
    analysis["temp_c"],
    bins=[-np.inf, 10, 20, 30, np.inf],
    labels=["freezing", "cold", "mild", "warm"],
)

In [ ]:
analysis.groupby("temp_band", observed=True).agg(
    hours=("pedestrians", "count"),
    mean_pedestrians=("pedestrians", "mean"),
    mean_temp_c=("temp_c", "mean"),
).round(1)

,hours,mean_pedestrians,mean_temp_c
temp_band,,,
freezing,1988,266.0,-3.9
cold,5407,496.6,4.9
mild,5182,700.5,16.0
warm,3464,1209.2,25.2


## 2.9 Linking two tables with `merge`

> **THE STORY** - One table says how many people crossed each day. Another says what the weather was each day. Link them by date: do warmer days have more people?
>
> **THE TOOL** - `.groupby(..., as_index=False)`, `.merge(on="date")`.

Both tables share a **common column**, `date`, and `merge` uses it to match each day's count with that same day's weather. `as_index=False` just keeps `date` as a normal column so the merge can find it.

In [ ]:
analysis["date"] = analysis["hour_beginning"].dt.normalize()     # the date, with the time stripped

In [ ]:
# Subtable 1: people per day
day_counts = analysis.groupby("date", as_index=False).agg(
    day_total=("pedestrians", "sum")
)
print(day_counts)

          date  day_total
0   2017-10-01      25687
1   2017-10-02      18028
2   2017-10-03      16580
3   2017-10-04      17533
4   2017-10-05      17980
..         ...        ...
664 2019-12-27      22663
665 2019-12-28      23892
666 2019-12-29      20829
667 2019-12-30       6190
668 2019-12-31      24071

[669 rows x 2 columns]


In [ ]:
# Subtable 2: weather per day
day_weather = analysis.groupby("date", as_index=False).agg(
    avg_temp=("temperature", "mean"),
)

print(day_weather)

          date   avg_temp
0   2017-10-01  57.166667
1   2017-10-02  59.041667
2   2017-10-03  58.583333
3   2017-10-04  62.500000
4   2017-10-05  71.208333
..         ...        ...
664 2019-12-27  48.375000
665 2019-12-28  45.625000
666 2019-12-29  40.083333
667 2019-12-30  40.541667
668 2019-12-31  40.458333

[669 rows x 2 columns]


In [ ]:
# Link them on the shared "date" column
daily = day_counts.merge(day_weather, on="date")
print(day_counts.shape, day_weather.shape, "->", daily.shape)    # same number of days, more columns

(669, 2) (669, 2) -> (669, 3)


In [ ]:
daily

,date,day_total,avg_temp
0,2017-10-01,25687,57.166667
1,2017-10-02,18028,59.041667
2,2017-10-03,16580,58.583333
3,2017-10-04,17533,62.500000
4,2017-10-05,17980,71.208333
...,...,...,...
664,2019-12-27,22663,48.375000
665,2019-12-28,23892,45.625000
666,2019-12-29,20829,40.083333
667,2019-12-30,6190,40.541667


In [ ]:
# Do warmer days have more people?
complete = daily.assign(warm_day=daily["avg_temp"] > daily["avg_temp"].mean())
complete

,date,day_total,avg_temp,warm_day
0,2017-10-01,25687,57.166667,True
1,2017-10-02,18028,59.041667,True
2,2017-10-03,16580,58.583333,True
3,2017-10-04,17533,62.500000,True
4,2017-10-05,17980,71.208333,True
...,...,...,...,...
664,2019-12-27,22663,48.375000,False
665,2019-12-28,23892,45.625000,False
666,2019-12-29,20829,40.083333,False
667,2019-12-30,6190,40.541667,False


In [ ]:
complete.groupby("warm_day")["day_total"].agg(["count", "mean"]).round(0)

,count,mean
warm_day,,
False,338,12386.0
True,331,20684.0


# Further Exercises


## Exercise 4 - Which way, hour by hour?

> **THE STORY** - Do people head towards Manhattan in the morning and back towards Brooklyn in the evening? Find the hour where each direction wins by the biggest margin.
>
> **THE TOOL** - `.groupby(...)`, `.mean()`, `.assign(...)`, `.idxmax()` / `.idxmin()`, `.loc`.

*Hint:* to average two columns at once, select them with double brackets after the group-by: `analysis.groupby("hour")[["a", "b"]]`. `idxmax()` returns the **hour** (the row label), not the value.

**Think:** does the answer fit the story of people walking into Manhattan in the morning and home in the evening?

In [1]:
# Your code here

## Exercise 5 - Rain at the same time of day

> **THE STORY** - Rainy hours looked quieter in 2.3, but was that the rain, or did rain just happen to fall at quiet hours of the day? Compare rainy and dry hours **only during the afternoon (12:00 to 17:59)**.
>
> **THE TOOL** - a True/False filter combined with `&`, `.assign(...)`, `.groupby(...)`, `.agg(...)`.

*Hint:* join two conditions with `&` and wrap each one in parentheses.

**Think:** which comparison is fairer, and why?

In [2]:
# Your code here

## Exercise 6 - Parts of the day

> **THE STORY** - Hour numbers are hard to read at a glance. Group the 24 hours into Night, Morning, Afternoon and Evening. Which part of the day is busiest, and what share of all pedestrians crosses in each?
>
> **THE TOOL** - `pd.cut`, `.groupby(..., observed=True)`, `.agg(...)`, `.assign(...)`, `.sort_values(...)`.

Use these definitions: Night = hours 0-5, Morning = 6-11, Afternoon = 12-17, Evening = 18-23.

*Hint:* `pd.cut` includes the right edge of each bin but **not** the left edge of the first one. If your first edge is `0`, what happens to hour 0? Try it and look for `NaN`.

In [3]:
# Your code here

## Exercise 7 - Weather groups

> **THE STORY** - Ten weather labels are too many to read at a glance. Bucket them into a few broad groups and ask: which group is busiest, and is its sample big enough to trust?
>
> **THE TOOL** - `.isna()`, `.fillna(...)`, `.apply(...)`, `.groupby(...).agg(...)`, `.sort_values(...)`.


*Hint:* `"cloudy" in label` tests whether a word appears inside a text label. It only works on text, which is why the missing values must be filled first.

**Think:** which group has the highest mean, and does it have enough hours to trust? Several groups mix daytime and night-time hours (`clear-night` is very quiet). How would that distort the comparison?

In [4]:
# Your code here

## Exercise 8 - Rainy days versus dry days

> **THE STORY** - Do days with rain have fewer pedestrians in total than days without rain?
>
> **THE TOOL** - `.assign(...)`, `.groupby("date", as_index=False).agg(...)`, `.merge(on="date")`, `.groupby(...).mean()`.

We already have `daily`, one row per date with `day_total` and `avg_temp` (from 2.9).


*Hint:* adding up a True/False column counts the `True` values.

**Think:** rainy days may also be colder. How could you check whether temperature, rather than rain, explains the gap?

In [5]:
# Your code here

## Key takeaways

The key transition to remember is:

**NumPy:** manipulate numerical arrays.

**pandas:** understand and manipulate labelled tables.

In a real data-science workflow, the two tools are often used together.


See you in Week 3: SQL Fundamentals.